# Extract and Store Entity Embeddings in Vector Database

This notebook extracts lifestyle factor (LSF) entities from 200 abstracts in `data/legacy/200Abstracts/train`:
1. **Ground Truth Entities**: Parsed from BRAT `.ann` files.
2. **Predicted Entities**: Extracted using the SpanNER model (without LLM escalation).
3. **Vector Embeddings via 9Router Proxy**: 4096-dimensional embeddings generated for all entities using your `qwen-embedding` combo model (or local fallback).
4. **Vector Database Storage**: Saved as abstract-indexed JSON (`lsf_entity_vector_db.json`) and persistent ChromaDB collection.


In [ ]:
# CELL 1: Environment Setup, Imports & Progress Logging Utility
import os
import sys
import glob
import json
import time
import torch
import numpy as np
from tqdm.std import tqdm
import chromadb
from dotenv import load_dotenv

# Add project root to sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

load_dotenv(os.path.join(PROJECT_ROOT, ".env"), override=True)

from src.data.dataset_converter import parse_ann_file
from src.architectures.hybrid_linkner.pipeline import LinkNERPipeline, DummyLLMProvider
from src.architectures.hybrid_linkner.novelty_detector import get_default_embedder

def log_progress(step_num, title, message):
    timestamp = time.strftime("%H:%M:%S")
    print(f"[{timestamp}] [STEP {step_num}: {title.upper()}] {message}")

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
log_progress(0, "Setup", f"Project Root: {PROJECT_ROOT}")
log_progress(0, "Setup", f"Compute Device: {device.upper()}")
log_progress(0, "Setup", f"Embedding Provider: {os.getenv('EMBEDDING_PROVIDER', '9router')} ({os.getenv('EMBEDDING_MODEL', 'qwen-embedding')})")


In [ ]:
# CELL 2: Step 1 - Extract Ground Truth Entities from BRAT .ann Files
DATA_DIR = os.path.join(PROJECT_ROOT, "data/legacy/200Abstracts/train")
ann_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.ann")))

log_progress(1, "Ground Truth", f"Found {len(ann_files)} BRAT .ann files in {DATA_DIR}")

gt_database = {}
total_gt_entities = 0
empty_ann_count = 0

for ann_path in ann_files:
    abstract_id = os.path.basename(ann_path).replace(".ann", "")
    parsed_ents = parse_ann_file(ann_path)
    
    formatted_ents = []
    for ent in parsed_ents:
        formatted_ents.append({
            "text": ent["text"],
            "label": ent["label"],
            "start_char": ent["start_char"],
            "end_char": ent["end_char"]
        })
        
    gt_database[abstract_id] = formatted_ents
    total_gt_entities += len(formatted_ents)
    if len(formatted_ents) == 0:
        empty_ann_count += 1

log_progress(1, "Ground Truth", f"Parsed Ground Truth for {len(gt_database)} abstracts.")
log_progress(1, "Ground Truth", f"Total GT Entities: {total_gt_entities} (Empty .ann files: {empty_ann_count})")


In [ ]:
# CELL 3: Step 2 - Predict Entities using SpanNER Model (Vectorized Batched Inference)
log_progress(2, "SpanNER Prediction", "Initializing SpanNER Model Pipeline...")
pipeline = LinkNERPipeline(llm_provider=DummyLLMProvider(verbose=False))

predicted_database = {}
total_pred_entities = 0
txt_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.txt")))

log_progress(2, "SpanNER Prediction", f"Running SpanNER vectorized inference on {len(txt_files)} abstract texts...")

for idx, txt_path in enumerate(tqdm(txt_files, desc="SpanNER Prediction"), 1):
    abstract_id = os.path.basename(txt_path).replace(".txt", "")
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read()
        
    # Setting uncertainty_threshold=1.0 guarantees local SpanNER predictions only without LLM calls
    preds = pipeline.predict(text, uncertainty_threshold=1.0)
    
    formatted_preds = []
    for p in preds:
        formatted_preds.append({
            "text": p["span_text"],
            "label": p["label"],
            "confidence": float(p["confidence"]),
            "start_char": p["start_char"],
            "end_char": p["end_char"]
        })
        
    predicted_database[abstract_id] = formatted_preds
    total_pred_entities += len(formatted_preds)
    
    if idx % 50 == 0 or idx == len(txt_files):
        log_progress(2, "SpanNER Prediction", f"Processed {idx}/{len(txt_files)} abstracts ({total_pred_entities} entities so far)")

log_progress(2, "SpanNER Prediction", f"SpanNER prediction complete. Total Predicted Entities: {total_pred_entities}")


In [ ]:
# CELL 4: Step 3 - Load 9Router Embedder & Generate Entity Vector Embeddings
log_progress(3, "Embedding Generation", "Initializing 9Router / Fallback Embedder...")
embedding_model = get_default_embedder(verbose=True)
log_progress(3, "Embedding Generation", f"Active Embedder: {embedding_model}")

# Collect all unique entity texts to avoid redundant encodings
all_texts = set()
for ents in gt_database.values():
    for e in ents:
        all_texts.add(e["text"])
for preds in predicted_database.values():
    for p in preds:
        all_texts.add(p["text"])

unique_text_list = sorted(list(all_texts))
log_progress(3, "Embedding Generation", f"Extracted {len(unique_text_list)} unique entity strings from GT & Predicted sets.")

# Batch encode entity texts
log_progress(3, "Embedding Generation", "Encoding entity texts into high-dimensional normalized vector embeddings...")
encoded_vectors = embedding_model.embed(unique_text_list)

embedding_dict = {text: vec.tolist() for text, vec in zip(unique_text_list, encoded_vectors)}
log_progress(3, "Embedding Generation", f"Successfully encoded {len(embedding_dict)} entities. Embedding Dimension: {encoded_vectors.shape[1]}")


In [ ]:
# CELL 5: Step 4 - Construct Abstract-Indexed Vector Database & Store in ChromaDB
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "data/vector_db")
os.makedirs(OUTPUT_DIR, exist_ok=True)

vector_db_json_path = os.path.join(OUTPUT_DIR, "lsf_entity_vector_db.json")
log_progress(4, "Vector DB Storage", f"Building Abstract-Indexed Vector Database...")

final_vector_db = {}
all_abstract_ids = sorted(list(set(gt_database.keys()).union(set(predicted_database.keys()))))

chroma_client = chromadb.PersistentClient(path=os.path.join(OUTPUT_DIR, "chroma_db"))
chroma_collection = chroma_client.get_or_create_collection(name="lsf_entities")

chroma_ids = []
chroma_embeddings = []
chroma_metadatas = []
chroma_documents = []

for abs_id in all_abstract_ids:
    gt_ents = gt_database.get(abs_id, [])
    pred_ents = predicted_database.get(abs_id, [])
    
    gt_records = []
    for idx, g in enumerate(gt_ents):
        emb = embedding_dict.get(g["text"], [])
        gt_records.append({
            "text": g["text"],
            "label": g["label"],
            "start_char": g["start_char"],
            "end_char": g["end_char"],
            "embedding": emb
        })
        # Add to ChromaDB
        chroma_ids.append(f"{abs_id}_gt_{idx}")
        chroma_embeddings.append(emb)
        chroma_metadatas.append({
            "abstract_id": abs_id,
            "source_type": "ground_truth",
            "label": g["label"],
            "start_char": g["start_char"],
            "end_char": g["end_char"]
        })
        chroma_documents.append(g["text"])
        
    pred_records = []
    for idx, p in enumerate(pred_ents):
        emb = embedding_dict.get(p["text"], [])
        pred_records.append({
            "text": p["text"],
            "label": p["label"],
            "confidence": p["confidence"],
            "start_char": p["start_char"],
            "end_char": p["end_char"],
            "embedding": emb
        })
        # Add to ChromaDB
        chroma_ids.append(f"{abs_id}_pred_{idx}")
        chroma_embeddings.append(emb)
        chroma_metadatas.append({
            "abstract_id": abs_id,
            "source_type": "predicted",
            "label": p["label"],
            "confidence": p["confidence"],
            "start_char": p["start_char"],
            "end_char": p["end_char"]
        })
        chroma_documents.append(p["text"])
        
    final_vector_db[abs_id] = {
        "abstract_id": abs_id,
        "ground_truth": gt_records,
        "predicted": pred_records
    }

# Save JSON Vector Database
with open(vector_db_json_path, "w", encoding="utf-8") as f:
    json.dump(final_vector_db, f, indent=2)
log_progress(4, "Vector DB Storage", f"Saved JSON Vector Database to: {vector_db_json_path}")

# Save ChromaDB Collection
if chroma_ids:
    chroma_collection.add(
        ids=chroma_ids,
        embeddings=chroma_embeddings,
        metadatas=chroma_metadatas,
        documents=chroma_documents
    )
log_progress(4, "Vector DB Storage", f"Persisted {len(chroma_ids)} entity vectors into ChromaDB collection ('lsf_entities')")


In [ ]:
# CELL 6: Step 5 - Verification & Vector DB Search Inspection
log_progress(5, "Verification", "Validating generated Vector Database...")

with open(vector_db_json_path, "r", encoding="utf-8") as f:
    loaded_db = json.load(f)

sample_key = list(loaded_db.keys())[0]
sample_entry = loaded_db[sample_key]

print("=" * 80)
print("                   VECTOR DATABASE VERIFICATION SUMMARY")
print("=" * 80)
print(f"Total Abstract Entries in DB : {len(loaded_db)}")
print(f"Sample Abstract ID           : {sample_key}")
print(f"GT Entities Count            : {len(sample_entry['ground_truth'])}")
print(f"Predicted Entities Count     : {len(sample_entry['predicted'])}")
if sample_entry['ground_truth']:
    gt_emb_dim = len(sample_entry['ground_truth'][0]['embedding'])
    print(f"GT Embedding Dimension       : {gt_emb_dim}")
if sample_entry['predicted']:
    pred_emb_dim = len(sample_entry['predicted'][0]['embedding'])
    print(f"Predicted Embedding Dimension: {pred_emb_dim}")

# ChromaDB Query Sample
query_results = chroma_collection.query(
    query_texts=["sleep disturbances"],
    n_results=3
)
print("\n--- ChromaDB Sample Query Test ('sleep disturbances') ---")
for doc, meta, dist in zip(query_results['documents'][0], query_results['metadatas'][0], query_results['distances'][0]):
    print(f"Match: '{doc}' | Label: {meta['label']} | Source: {meta['source_type']} | Abstract: {meta['abstract_id']} | Distance: {dist:.4f}")
print("=" * 80)
log_progress(5, "Verification", "Verification complete! Vector Database ready for delivery.")
